In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Đọc dữ liệu
df = pd.read_csv('customer_churn.csv')

# Xử lý giá trị thiếu (Missing Values)
# Cột TotalCharges có thể trống ở khách hàng mới → thay bằng 0 hoặc giá trị trung bình
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')  # chuyển về dạng số
df['TotalCharges'].fillna(0, inplace=True)

# Kiểm tra các cột khác có giá trị thiếu
print("Missing values:\n", df.isnull().sum())

# Thay thế giá trị thiếu cho các cột khác nếu có
df.fillna({
    'Gender': df['Gender'].mode()[0],
    'PaymentMethod': df['PaymentMethod'].mode()[0],
    'Contract': df['Contract'].mode()[0]
}, inplace=True)

# 2️⃣ Xử lý ngoại lệ (Outliers)
# Dùng IQR cho các biến số như MonthlyCharges, Tenure
for col in ['MonthlyCharges', 'Tenure', 'TotalCharges']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

# 3️⃣ Mã hóa dữ liệu phân loại (Categorical Encoding)
# Chuyển Yes/No thành 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# One-hot encoding cho Contract và PaymentMethod
df = pd.get_dummies(df, columns=['Contract', 'PaymentMethod'], drop_first=True)

# 4️⃣ Chuẩn hóa biến số (Feature Scaling)
scaler = StandardScaler()
num_cols = ['Tenure', 'MonthlyCharges', 'TotalCharges']
df[num_cols] = scaler.fit_transform(df[num_cols])

# 5️⃣ Kiểm tra kết quả
print(df.head())
